**Historical engineering notebook (Phase 7 smoke).** Not the official 420-case evaluation. Official evaluation: Phase 15 (benchmark) and Phase 16 (judge).


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# V2 Phase 7 — Colab GPU smoke (Qwen3-8B)

**Before running:** Runtime → Change runtime type → **GPU**.

## Setup instructions

Your GitHub repo has **V2/** at the repository root:

```text
repo/
└── V2/                ← MSc project code — Colab uses THIS folder
```

1. **Push latest changes** to branch `main`
2. Open this notebook with **GPU** runtime
3. Run all cells — cell 1 clones the repo, then **`cd` into `V2/`**

No need to upload anything to Google Drive for source code.

**Outputs:** `V2/results/config/phase7_smoke_test.json` (after smoke run)

## 1. Clone GitHub repo and enter V2

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/syedsafiullah777/CAPSTONE--RAG-WITH-UNCERTAINITY-QUANTIFICATION-.git'
BRANCH = 'main'
CLONE_DIR = Path('/content/capstone-rag')

if CLONE_DIR.exists():
    !rm -rf {CLONE_DIR}

print('Cloning branch:', BRANCH)
result = subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(CLONE_DIR)],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'git clone failed. Push V2/ to GitHub on branch {BRANCH!r} first.')

print('Repo root contents:', [p.name for p in CLONE_DIR.iterdir()])
# GitHub layout: repository root contains V2/ — we work inside V2/
V2_ROOT = CLONE_DIR / 'V2'
if not V2_ROOT.is_dir():
    raise FileNotFoundError(
        f'Missing V2/ folder at {V2_ROOT}. '
        'Your repo should have V2/ at the top level.'
    )
if not (V2_ROOT / 'scripts' / 'smoke_generate.py').is_file():
    raise FileNotFoundError(f'Invalid V2 root: {V2_ROOT}')

os.chdir(V2_ROOT)
sys.path.insert(0, str(V2_ROOT))
print('OK — working in V2_ROOT:', V2_ROOT)
!git -C {CLONE_DIR} log -1 --oneline

## 2. Install dependencies

In [3]:
!pip -q install -r requirements.txt
!pip -q install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/6

## 3. Run smoke test (llama_cpp on Colab GPU)

In [ ]:
!PYTHONPATH=. python scripts/smoke_generate.py --backend llama_cpp --notebook notebooks/colab_phase7_smoke.ipynb

In [ ]:
# Fallback only if llama_cpp fails:
# !PYTHONPATH=. python scripts/smoke_generate.py --backend transformers --notebook notebooks/colab_phase7_smoke.ipynb

## 4. Check results

In [ ]:
import json
from pathlib import Path

fp = Path('results/config/phase7_runtime_fingerprint.json')
smoke = Path('results/config/phase7_smoke_test.json')
print('fingerprint:', fp.is_file())
print('smoke_test:', smoke.is_file())
if smoke.is_file():
    data = json.loads(smoke.read_text())
    print('status:', data.get('status'))
    print('actual:', repr(data.get('actual')))

## 5. (Optional) Copy results to Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')
dest = Path('/content/drive/MyDrive/MSc-RAG/configs/phase7')
dest.mkdir(parents=True, exist_ok=True)
for name in ('phase7_runtime_fingerprint.json', 'phase7_smoke_test.json'):
    src = Path('results/config') / name
    if src.is_file():
        shutil.copy2(src, dest / name)
        print('copied', name)

In [6]:
%cd /content/capstone-rag
!git pull

/content/capstone-rag
Already up to date.


In [7]:
# Verify gpu
import torch

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(
        torch.cuda.get_device_properties(0).total_memory / 1e9, 2
    ))

CUDA: True
GPU: Tesla T4
VRAM GB: 15.64


In [8]:
%cd /content/capstone-rag
!git pull
!PYTHONPATH=. python V2/scripts/smoke_single_agent.py --backend llama_cpp --limit 3

/content/capstone-rag
Already up to date.
2026-08-23T12:02:52 | INFO | run_id=phase8_20260823T120228Z_8cf4510a | phase=phase8 | model=- | device=- | architecture=single_agent | question_id=- | Running question_id=finqa_test_1000
modules.json: 100% 349/349 [00:00<00:00, 1.25MB/s]
config_sentence_transformers.json: 100% 124/124 [00:00<00:00, 560kB/s]
README.md: 100% 94.8k/94.8k [00:00<00:00, 91.5MB/s]
sentence_bert_config.json: 100% 52.0/52.0 [00:00<00:00, 220kB/s]
config.json: 100% 743/743 [00:00<00:00, 3.90MB/s]

model.safetensors: downloading bytes:  59% 79.4M/133M [00:02<00:00, 62.2MB/s, 6.56MB/s  ]
model.safetensors: downloading bytes: 100% 86.7M/86.7M [00:02<00:00, 39.3MB/s, 8.05MB/s  ]
model.safetensors: reconstructing file: 100% 133M/133M [00:02<00:00, 60.4MB/s, 12.6MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 17519.55it/s]
tokenizer_config.json: 100% 366/366 [00:00<00:00, 1.74MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 93.9MB/s]
tokenizer.json: 100% 711k/711k [00:00<00: